# TP2 - Informe tecnico

Sistema de Deteccion y Clasificacion de Razas de Perros — IA 5.2 Computer Vision.

## Equipo
- Alumno 1 :
- Alumno 2 :

## 1. Explicacion completa del pipeline

TO-DO: describir el flujo Embeddings -> Busqueda por similitud -> Clasificacion -> Deteccion -> Pipeline completo,
y como se integran los componentes (base vectorial, modelos, YOLO, aplicacion Gradio).

## 2. Dataset

TO-DO: distribucion de clases, cantidad de imagenes por raza, definicion de splits
(train/valid/test) y conjunto independiente de evaluacion.

## 3. Preprocesamiento

TO-DO: tecnicas aplicadas (resize, normalizacion, data augmentation, filtrado) y su justificacion.

## 4. Justificacion de los modelos elegidos

TO-DO: modelo de embeddings baseline (Etapa 1), ResNet18 fine-tuned y CNN custom (Etapa 2),
YOLO (Etapa 3). Analizar trade-offs: precision, velocidad de inferencia, consumo de memoria
y complejidad computacional.

## 5. Proceso de entrenamiento e hiperparametros

TO-DO: proceso de fine-tuning, hiperparametros utilizados (learning rate, batch size, epochs,
optimizador, scheduler), curvas de entrenamiento.

## 6. Resultados obtenidos

TO-DO:
- Etapa 1: NDCG@10 y justificacion del resultado.
- Etapa 2: accuracy, precision, recall, specificity, F1, matriz de confusion.

## 7. Comparacion entre enfoques

TO-DO: busqueda por similitud vs clasificacion supervisada; ResNet18 fine-tuned vs CNN custom.

## 8. Problemas encontrados y soluciones implementadas

TO-DO.

## 9. Modificaciones fuera de las funciones indicadas

TO-DO: justificar debidamente cualquier cambio realizado fuera de las funciones
indicadas en cada etapa (si no hubo, indicarlo).

**Nota sobre `data/embeddings.json` y `models/*.pth`:**

Estos archivos no se versionan en git (ver `.gitignore`) por ser
artefactos generados, regenerables ejecutando los scripts
correspondientes (`scripts/build_index.py` para la base vectorial,
`scripts/train_classifier.py` para los checkpoints de modelos), y por
su tamaño (la base vectorial completa supera varios MB con ~8000
embeddings de 512 dimensiones). Para reproducir el estado del sistema
desde cero: `python scripts/download_dataset.py`, seguido de
`python scripts/build_index.py --split train` y
`python scripts/train_classifier.py --model resnet18_finetuned`.

### Modificación justificada: `PgVectorEmbeddingStore.search()`

**Archivo:** `src/lib/storage/pgvector_store.py`

**Problema encontrado:** al ejecutar `search_similar_images` con el backend
pgvector, la consulta SQL fallaba con el error:

**Causa:** el operador `<=>` de la extensión pgvector requiere que ambos
operandos sean del tipo `vector`. El parámetro de la consulta (`%s`) se
pasa desde Python como una lista de floats, que psycopg serializa como
`double precision[]` (un array genérico de PostgreSQL) en lugar de
`vector`, a pesar de que `register_vector(self.conn)` se invoca en el
constructor de la clase. PostgreSQL no realiza el cast implícito
necesario para el operador `<=>` en este contexto.

**Solución aplicada:** se agregó un cast explícito a `vector` en la
clausula `ORDER BY` de la consulta:

```sql
-- Antes:
ORDER BY embedding <=> %s
-- Despues:
ORDER BY embedding <=> %s::vector
```

**Alcance de la modificación:** un único cast de tipo en una consulta
SQL, sin alterar la firma del metodo, su comportamiento esperado, ni
la interfaz `EmbeddingStoreProtocol`. No afecta a `extract_embedding`,
`search_similar_images` ni `predict_breed_from_neighbors` (funciones de
Etapa 1 implementadas por el estudiante), que permanecen sin cambios.